# 14 - Scatter Gols Marcados vs Sofridos e Bump Chart de Ranking

- Scatter: gols marcados vs sofridos por posição final
- Bump chart: evolução do ranking ao longo das rodadas (temporada específica)
- Placares mais comuns: mudaram ao longo das décadas?

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

df = pd.read_csv('../data/raw/campeonato-brasileiro-full.csv')
df['data'] = pd.to_datetime(df['data'], format='%d/%m/%Y')
df['ano'] = df['data'].dt.year
df = df.dropna(subset=['rodata'])
df['rodata'] = df['rodata'].astype(int)
df = df[df['ano'] >= 2003].copy()

def classificacao_completa(df_season):
    teams = set(df_season['mandante'].unique()) | set(df_season['visitante'].unique())
    pontos = {t: 0 for t in teams}
    vitorias = {t: 0 for t in teams}
    saldo = {t: 0 for t in teams}
    gols_pro = {t: 0 for t in teams}
    gols_contra = {t: 0 for t in teams}
    for _, jogo in df_season.iterrows():
        m, v = jogo['mandante'], jogo['visitante']
        gm, gv = jogo['mandante_Placar'], jogo['visitante_Placar']
        if pd.isna(gm) or pd.isna(gv): continue
        gm, gv = int(gm), int(gv)
        gols_pro[m] += gm; gols_contra[m] += gv
        gols_pro[v] += gv; gols_contra[v] += gm
        saldo[m] += gm - gv; saldo[v] += gv - gm
        if gm > gv: pontos[m] += 3; vitorias[m] += 1
        elif gm < gv: pontos[v] += 3; vitorias[v] += 1
        else: pontos[m] += 1; pontos[v] += 1
    ranking = sorted(teams, key=lambda t: (pontos[t], vitorias[t], saldo[t]), reverse=True)
    return [{' time': t, 'pontos': pontos[t], 'posicao': i+1, 'gols_pro': gols_pro[t],
             'gols_contra': gols_contra[t], 'saldo': saldo[t]} for i, t in enumerate(ranking)]

print('Dados carregados')

Dados carregados


In [2]:
# SCATTER: Gols marcados vs sofridos colorido por posição
todos = []
for ano in sorted(df['ano'].unique()):
    df_s = df[df['ano'] == ano]
    if df_s['rodata'].max() < 30: continue
    classif = classificacao_completa(df_s)
    for c in classif:
        c['ano'] = ano
        c['time'] = c.pop(' time')
    todos.extend(classif)

df_all = pd.DataFrame(todos)

# Faixa de posição
def faixa(p):
    if p == 1: return 'Campe\u00e3o'
    elif p <= 4: return 'Libertadores (2\u00ba-4\u00ba)'
    elif p <= 10: return 'Meio (5\u00ba-10\u00ba)'
    elif p <= 16: return 'Baixo (11\u00ba-16\u00ba)'
    else: return 'Z4 (17\u00ba+)'

df_all['faixa'] = df_all['posicao'].apply(faixa)
df_all['faixa'] = pd.Categorical(df_all['faixa'], 
    ['Campe\u00e3o', 'Libertadores (2\u00ba-4\u00ba)', 'Meio (5\u00ba-10\u00ba)', 'Baixo (11\u00ba-16\u00ba)', 'Z4 (17\u00ba+)'])

fig = px.scatter(df_all, x='gols_pro', y='gols_contra',
                 color='faixa',
                 color_discrete_map={'Campe\u00e3o': '#f1c40f', 'Libertadores (2\u00ba-4\u00ba)': '#27ae60',
                                     'Meio (5\u00ba-10\u00ba)': '#3498db', 'Baixo (11\u00ba-16\u00ba)': '#95a5a6',
                                     'Z4 (17\u00ba+)': '#e74c3c'},
                 hover_data=['time', 'ano', 'pontos', 'posicao'],
                 opacity=0.6,
                 title='Gols Marcados vs Sofridos por Posi\u00e7\u00e3o Final<br><sup>Brasileir\u00e3o (2003-presente)</sup>',
                 labels={'gols_pro': 'Gols Marcados', 'gols_contra': 'Gols Sofridos', 'faixa': 'Classifica\u00e7\u00e3o'})

# Linha diagonal (saldo = 0)
max_g = max(df_all['gols_pro'].max(), df_all['gols_contra'].max())
fig.add_trace(go.Scatter(x=[20, max_g], y=[20, max_g],
                          mode='lines', line=dict(color='gray', dash='dot'),
                          name='Saldo = 0', showlegend=False))

fig.update_layout(template='plotly_white', width=800, height=650,
                  legend=dict(x=0.01, y=0.99))

fig.show()
fig.write_html('../charts/scatter_gols.html', include_plotlyjs='cdn')
print('Gr\u00e1fico exportado: charts/scatter_gols.html')

Gráfico exportado: charts/scatter_gols.html


In [3]:
# BUMP CHART: evolução do ranking ao longo das rodadas (última temporada completa)
ultimo_ano = sorted(df['ano'].unique())[-1]
df_season = df[df['ano'] == ultimo_ano]
max_rod = df_season['rodata'].max()

if max_rod < 30:
    ultimo_ano = sorted(df['ano'].unique())[-2]
    df_season = df[df['ano'] == ultimo_ano]
    max_rod = df_season['rodata'].max()

print(f'Bump chart para temporada {ultimo_ano} ({max_rod} rodadas)')

# Classificacao rodada a rodada
teams = set(df_season['mandante'].unique()) | set(df_season['visitante'].unique())
pontos = {t: 0 for t in teams}
vitorias = {t: 0 for t in teams}
saldo = {t: 0 for t in teams}

bump_data = []

for rodada in range(1, int(max_rod) + 1):
    jogos = df_season[df_season['rodata'] == rodada]
    for _, jogo in jogos.iterrows():
        m, v = jogo['mandante'], jogo['visitante']
        gm, gv = jogo['mandante_Placar'], jogo['visitante_Placar']
        if pd.isna(gm) or pd.isna(gv): continue
        gm, gv = int(gm), int(gv)
        saldo[m] += gm - gv; saldo[v] += gv - gm
        if gm > gv: pontos[m] += 3; vitorias[m] += 1
        elif gm < gv: pontos[v] += 3; vitorias[v] += 1
        else: pontos[m] += 1; pontos[v] += 1
    
    ranking = sorted(teams, key=lambda t: (pontos[t], vitorias[t], saldo[t]), reverse=True)
    for i, t in enumerate(ranking):
        bump_data.append({'rodada': rodada, 'time': t, 'posicao': i+1, 'pontos_acum': pontos[t]})

df_bump = pd.DataFrame(bump_data)

Bump chart para temporada 2024 (38 rodadas)


In [4]:
# Selecionar top 10 + bottom 4 para n\u00e3o poluir
final = df_bump[df_bump['rodada'] == max_rod].sort_values('posicao')
top_times = list(final.head(10)['time']) + list(final.tail(4)['time'])

df_bump_sel = df_bump[df_bump['time'].isin(top_times)]

# Cores
cores_bump = {}
for _, row in final.iterrows():
    if row['posicao'] == 1: cores_bump[row['time']] = '#f1c40f'
    elif row['posicao'] <= 4: cores_bump[row['time']] = '#27ae60'
    elif row['posicao'] <= 10: cores_bump[row['time']] = '#3498db'
    elif row['posicao'] > len(teams) - 4: cores_bump[row['time']] = '#e74c3c'
    else: cores_bump[row['time']] = '#95a5a6'

fig2 = go.Figure()

for time in top_times:
    dt = df_bump_sel[df_bump_sel['time'] == time]
    fig2.add_trace(go.Scatter(
        x=dt['rodada'], y=dt['posicao'],
        mode='lines',
        name=time,
        line=dict(color=cores_bump.get(time, '#95a5a6'), width=2),
        hovertemplate=f'{time}<br>Rodada %{{x}}: %{{y}}\u00ba<extra></extra>'
    ))

fig2.update_yaxes(autorange='reversed', title='Posi\u00e7\u00e3o')
fig2.add_hrect(y0=0.5, y1=4.5, fillcolor='green', opacity=0.03)
fig2.add_hrect(y0=len(teams)-3.5, y1=len(teams)+0.5, fillcolor='red', opacity=0.03)

fig2.update_layout(
    title=f'Bump Chart: Evolu\u00e7\u00e3o do Ranking - Brasileir\u00e3o {ultimo_ano}<br><sup>Top 10 + Z4 | Verde=Libertadores, Vermelho=Rebaixamento</sup>',
    xaxis_title='Rodada',
    template='plotly_white',
    width=1100,
    height=650,
    legend=dict(x=1.02, y=1)
)

fig2.show()
fig2.write_html('../charts/bump_chart.html', include_plotlyjs='cdn')
print('Gr\u00e1fico exportado: charts/bump_chart.html')

Gráfico exportado: charts/bump_chart.html


In [5]:
# PLACARES MAIS COMUNS: mudan\u00e7a ao longo das d\u00e9cadas?
df['placar'] = df['mandante_Placar'].astype(int).astype(str) + 'x' + df['visitante_Placar'].astype(int).astype(str)
df['periodo'] = df['ano'].apply(lambda a: '2003-2009' if a <= 2009 else ('2010-2016' if a <= 2016 else '2017-2024'))

top_placares = df['placar'].value_counts().head(8).index.tolist()

placar_periodo = df[df['placar'].isin(top_placares)].groupby(['periodo', 'placar']).size().reset_index(name='jogos')
# Normalizar por total de jogos do per\u00edodo
total_periodo = df.groupby('periodo').size().reset_index(name='total')
placar_periodo = placar_periodo.merge(total_periodo, on='periodo')
placar_periodo['pct'] = placar_periodo['jogos'] / placar_periodo['total'] * 100

fig3 = px.bar(placar_periodo, x='placar', y='pct', color='periodo',
              barmode='group',
              title='Placares Mais Comuns por Per\u00edodo<br><sup>O jogo ficou mais ou menos el\u00e1stico?</sup>',
              labels={'placar': 'Placar', 'pct': '% dos Jogos', 'periodo': 'Per\u00edodo'},
              color_discrete_sequence=['#3498db', '#2ecc71', '#e74c3c'])
fig3.update_layout(template='plotly_white', width=900, height=450)

fig3.show()
fig3.write_html('../charts/placares_decadas.html', include_plotlyjs='cdn')
print('Gr\u00e1fico exportado: charts/placares_decadas.html')

Gráfico exportado: charts/placares_decadas.html
